# Phase 4 — Hybrid System

Pipeline: **rule-based jargon substitution → T5-small generation**

Inputs:
- `data/processed/` — Phase 1 Arrow test split
- `data/medical_dict.json` — Phase 2 dictionary
- `CHECKPOINT_PATH` — Phase 3 T5-small best checkpoint on Google Drive

Outputs:
- `predictions/hybrid.jsonl` — 1,046 prediction records
- `results/metrics.csv` — updated with hybrid row

## Section 1 — Setup

In [ ]:
# Install dependencies (skip if already installed)
import importlib, subprocess, sys

def _install(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_install('transformers')
_install('datasets')
_install('sacrebleu')
_install('textstat')
_install('easse @ git+https://github.com/feralvam/easse.git', 'easse')
_install('spacy')

print('Dependencies ready.')

In [ ]:
import os, sys, json, random
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print('Seed set:', RANDOM_SEED)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Mount Google Drive
from google.colab import drive, userdata
drive.mount('/drive')

# *** SET THESE TO MATCH YOUR DRIVE LAYOUT ***
CHECKPOINT_PATH  = '/drive/MyDrive/NLP_Project/checkpoints/t5_small/checkpoint-10460'
DRIVE_DICT_PATH  = '/drive/MyDrive/NLP_Project/medical_dict.json'
DRIVE_DATA_PATH  = '/drive/MyDrive/NLP_Project/processed'

# Clone or update repo using GitHub token from Colab secrets
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/IbrahimHanafy2222/NLP-Project.git'
PROJECT_ROOT = '/content/NLP-Project'

import subprocess as _sp

def _is_git_repo(path):
    try:
        _sp.check_call(['git', '-C', path, 'rev-parse', '--git-dir'],
                       stdout=_sp.DEVNULL, stderr=_sp.DEVNULL)
        return True
    except _sp.CalledProcessError:
        return False

if _is_git_repo(PROJECT_ROOT):
    _sp.check_call(['git', '-C', PROJECT_ROOT, 'pull'])
    print('Repo updated.')
else:
    import shutil as _sh
    if os.path.exists(PROJECT_ROOT):
        _sh.rmtree(PROJECT_ROOT)
        print('Removed broken directory.')
    _sp.check_call(['git', 'clone', REPO_URL, PROJECT_ROOT])
    print('Repo cloned.')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Working dir:', os.getcwd())

## Section 2 — Load Data

In [ ]:
dataset = load_from_disk('data/processed/')
test_split = dataset['test']

sources    = test_split['source']
references = test_split['target']

print(f'Test split size: {len(test_split)}')
assert len(test_split) == 1046, f'Expected 1046, got {len(test_split)}'
print('Sample source:', sources[0])
print('Sample reference:', references[0])

## Section 3 — Load Pipeline

In [ ]:
from src.hybrid import HybridPipeline

pipeline = HybridPipeline(
    dict_path='data/medical_dict.json',
    checkpoint_path=CHECKPOINT_PATH,
)

print('Model device:', pipeline.device)
print('Dictionary entries:', len(pipeline.substituter.dictionary))

# Quick sanity check
_sample = pipeline.simplify('The patient presented with myocardial infarction.')
print('Sanity check output:', _sample)

## Section 4 — Inference

In [ ]:
print(f'Running inference on {len(sources)} sentences...')
predictions = pipeline.batch_simplify(list(sources), batch_size=8)

print(f'Done. Generated {len(predictions)} predictions.')
print()
print('--- First 3 examples ---')
for i in range(3):
    print(f'[{i+1}] SOURCE:     {sources[i]}')
    print(f'    PREDICTION: {predictions[i]}')
    print(f'    REFERENCE:  {references[i]}')
    print()

## Section 5 — Save Predictions

In [ ]:
os.makedirs('predictions', exist_ok=True)
output_path = 'predictions/hybrid.jsonl'

with open(output_path, 'w', encoding='utf-8') as f:
    for src, pred, ref in zip(sources, predictions, references):
        # source = ORIGINAL text (pre-substitution) for fair metric comparison
        record = {'source': src, 'prediction': pred, 'reference': ref}
        f.write(json.dumps(record) + '\n')

with open(output_path) as f:
    line_count = sum(1 for _ in f)

print(f'Saved {line_count} records to {output_path}')
assert line_count == len(sources), f'Mismatch: {line_count} lines vs {len(sources)} sources'

## Section 6 — Evaluate

In [ ]:
from src.metrics import compute_sari, compute_bleu, compute_fkgl

sources_list     = list(sources)
references_list  = list(references)

sari        = compute_sari(sources_list, predictions, references_list)
bleu        = compute_bleu(predictions, references_list)
fkgl_input  = compute_fkgl(sources_list)
fkgl_output = compute_fkgl(predictions)
fkgl_delta  = fkgl_output - fkgl_input

print(f'SARI:        {sari:.4f}')
print(f'BLEU:        {bleu:.4f}')
print(f'FKGL input:  {fkgl_input:.4f}')
print(f'FKGL output: {fkgl_output:.4f}')
print(f'FKGL delta:  {fkgl_delta:.4f}  (negative = simpler)')

## Section 7 — Log Metrics

In [ ]:
metrics_path = 'results/metrics.csv'
df = pd.read_csv(metrics_path)

# Remove stale hybrid row if re-running
df = df[df['system'] != 'hybrid']

hybrid_row = pd.DataFrame([{
    'system':      'hybrid',
    'sari':        round(sari,        4),
    'bleu':        round(bleu,        4),
    'fkgl_input':  round(fkgl_input,  4),
    'fkgl_output': round(fkgl_output, 4),
    'fkgl_delta':  round(fkgl_delta,  4),
}])

df = pd.concat([df, hybrid_row], ignore_index=True)
df.to_csv(metrics_path, index=False)
print(f'Updated {metrics_path}')
print(df.to_string(index=False))

## Section 8 — Comparison Table

In [ ]:
df_all = pd.read_csv('results/metrics.csv')

# Pretty comparison table
display_order = ['rule_based', 't5_small', 'scifive', 'hybrid']
df_display = df_all.set_index('system').reindex(
    [s for s in display_order if s in df_all['system'].values]
).reset_index()

print('=' * 70)
print('RESULTS: All Systems × All Metrics')
print('=' * 70)
print(df_display.to_string(index=False))
print('=' * 70)

t5_sari     = df_all.loc[df_all['system'] == 't5_small', 'sari'].values[0]
hybrid_sari = df_all.loc[df_all['system'] == 'hybrid',   'sari'].values[0]
delta       = hybrid_sari - t5_sari

direction = 'IMPROVEMENT' if delta > 0 else 'REGRESSION'
print(f'\nHybrid vs T5-small SARI: {delta:+.4f} ({direction})')
if delta <= 0:
    print('Note: Hybrid did not improve SARI. Reporting result honestly (Constitution III).')

## Section 9 — Verification Assertions

In [ ]:
# quickstart.md verification block
with open('predictions/hybrid.jsonl') as f:
    records = [json.loads(line) for line in f]

assert len(records) == 1046, f'Expected 1046 records, got {len(records)}'
assert all('source' in r and 'prediction' in r and 'reference' in r for r in records), \
    'Missing required fields in prediction records'

df_verify = pd.read_csv('results/metrics.csv')
hybrid_row_verify = df_verify[df_verify['system'] == 'hybrid'].iloc[0]
assert hybrid_row_verify['fkgl_delta'] < 0, \
    f'FKGL did not decrease: delta={hybrid_row_verify["fkgl_delta"]}'

print('ALL ASSERTIONS PASSED')
print(f'  Records: {len(records)}')
print(f'  SARI: {hybrid_row_verify["sari"]}')
print(f'  FKGL delta: {hybrid_row_verify["fkgl_delta"]} (< 0 ✓)')